In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import asyncio
import json
import os
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import Audio
import pyquist as pq
from scipy.signal import lfilter
import icm_plotly
from icm_plotly import RED, BLUE, IRON, STEEL

In [ ]:
# hide
# autorun
with open("../assets/mozart-dice.json") as file:
    DICE = json.load(file)
TABLE = {int(s): bars for s, bars in DICE["table"].items()}   # sum of the dice -> 16 bars
BARS = {int(b): notes for b, notes in DICE["bars"].items()}   # bar number -> its notes
SUMS = list(range(2, 13))
P_SUM = {s: (6 - abs(s - 7)) / 36 for s in SUMS}              # P(S = s) for two dice
SR = 44100
EIGHTH = 0.3                                                  # seconds per eighth note
HOP = int(3 * EIGHTH * SR)                                    # samples per 3/8 bar
TAIL = int(1.5 * SR)                                          # room for the strings to ring
ROLLS0 = [7, 5, 8, 6, 9, 7, 4, 7, 6, 10, 7, 8, 3, 7, 8, 7]    # the waltz on the page at first
PITCHES = [n["pitch"] for notes in BARS.values() for n in notes]
P_LO, P_HI = min(PITCHES) - 2, max(PITCHES) + 2


def pluck(pitch, duration, f_s=SR):
    """The tuned Karplus-Strong plucked string from this chapter's other examples."""
    P = f_s / pq.helper.pitch_to_frequency(pitch)
    M = int(P - 0.6)
    D = P - 0.5 - M
    c = (1 - D) / (1 + D)
    g = 10 ** (-3 * P / (f_s * duration))
    x = np.zeros(int(duration * f_s))
    x[:M] = np.random.uniform(-1.0, 1.0, M)
    a = np.zeros(M + 3)
    a[:2] = [1.0, c]
    a[M:] -= 0.5 * g * np.array([c, 1 + c, 1.0])
    return lfilter([1.0, c], a, x)


def render_bar(bar, cache={}):
    """The audio of one bar, including its ringing tail (cached: each bar renders once)."""
    if bar not in cache:
        x = np.zeros(HOP + TAIL)
        for note in BARS[bar]:
            y = 0.5 * pluck(note["pitch"], 0.6 + EIGHTH * note["duration"])
            start = int(note["onset"] * EIGHTH * SR)
            x[start:start + len(y)] += y[:len(x) - start]
        cache[bar] = x
    return cache[bar]


def piano_roll(rolls):
    """Line segments for the notes of the treble and the bass, in measures."""
    lines = {"treble": ([], []), "bass": ([], [])}
    for k, s in enumerate(rolls):
        for note in BARS[TABLE[s][k]]:
            x0 = k + note["onset"] / 3
            xs, ys = lines.get(note["staff"], lines["treble"])
            xs += [x0, x0 + 0.95 * note["duration"] / 3, None]
            ys += [note["pitch"], note["pitch"], None]
    return lines


def figure():
    lines = piano_roll(ROLLS0)
    fig = go.Figure()
    bar_x = [x for k in range(17) for x in (k, k, None)]
    bar_y = [y for k in range(17) for y in (P_LO, P_HI, None)]
    fig.add_scatter(x=bar_x, y=bar_y, mode="lines", line=dict(color=STEEL, width=1),
                    hoverinfo="skip")
    fig.add_scatter(x=lines["treble"][0], y=lines["treble"][1], mode="lines",
                    line=dict(color=RED, width=6), hoverinfo="skip")
    fig.add_scatter(x=lines["bass"][0], y=lines["bass"][1], mode="lines",
                    line=dict(color=BLUE, width=6), hoverinfo="skip")
    fig.update_xaxes(range=[0, 16], title_text="Measure", tickvals=np.arange(16) + 0.5,
                     ticktext=[str(k + 1) for k in range(16)], fixedrange=True)
    fig.update_yaxes(range=[P_LO, P_HI], title_text="MIDI pitch", fixedrange=True)
    fig.update_layout(showlegend=False, height=320)
    return fig


def controls(fig):
    rolls = list(ROLLS0)
    buttons = {}
    cells = [widgets.HTML("")]
    cells += [widgets.HTML(f"<div style='text-align:center'><b>{k + 1}</b></div>") for k in range(16)]
    for s in SUMS:
        cells.append(widgets.HTML(f"<div style='text-align:right; padding-right:6px'>"
                                  f"<b>{s}</b></div>"))
        for k in range(16):
            button = widgets.Button(description=str(TABLE[s][k]), tooltip=f"Measure {k + 1}, "
                                    f"sum {s}: bar {TABLE[s][k]}",
                                    layout=widgets.Layout(width="auto", height="24px", padding="0"))
            button.style.font_size = "11px"
            buttons[s, k] = button
            cells.append(button)
    grid = widgets.GridBox(cells, layout=widgets.Layout(
        grid_template_columns="36px repeat(16, minmax(0, 1fr))", grid_gap="2px", width="100%"))
    roll = widgets.Button(description="Roll the dice", icon="random")
    readout = widgets.HTML()
    out = widgets.Output()
    pending = []

    # the defaults snapshot the helpers; the page's notebooks share one kernel
    def paint(buttons=buttons, P_SUM=P_SUM):
        for (s, k), button in buttons.items():
            chosen = rolls[k] == s
            button.style.button_color = RED if chosen else "#f2f2f2"
            button.style.text_color = "white" if chosen else "#444444"
        p = np.prod([P_SUM[s] for s in rolls])
        readout.value = (f"<span style='font-size:0.9em'>Sums: {' '.join(map(str, rolls))} "
                         f"&nbsp;·&nbsp; probability of rolling exactly these sums: "
                         f"1 in {1 / p:,.0f}</span>")

    def redraw(piano_roll=piano_roll):
        lines = piano_roll(rolls)
        with fig.batch_update():
            fig.data[1].x, fig.data[1].y = lines["treble"]
            fig.data[2].x, fig.data[2].y = lines["bass"]

    def render(TABLE=TABLE, render_bar=render_bar, HOP=HOP, TAIL=TAIL, SR=SR):
        x = np.zeros(16 * HOP + TAIL)
        for k, s in enumerate(rolls):
            x[k * HOP:k * HOP + HOP + TAIL] += render_bar(TABLE[s][k])
        x *= 0.125 / np.abs(x).max()          # about -18 dBFS, a safe level
        audio = Audio(x.astype(np.float32), rate=SR, normalize=False)
        data, metadata = get_ipython().display_formatter.format(audio)
        # one assignment swaps the old card for the new one in place
        out.outputs = ({"output_type": "display_data", "data": data, "metadata": metadata},)

    async def settle():
        await asyncio.sleep(0.3)
        pending.clear()
        render()

    def changed():
        paint()
        redraw()
        if pending:
            pending.pop().cancel()
        pending.append(asyncio.ensure_future(settle()))

    def choose(s, k):
        rolls[k] = s
        changed()

    def roll_dice(_):
        for k in range(16):
            rolls[k] = int(np.random.randint(1, 7) + np.random.randint(1, 7))
        changed()

    for (s, k), button in buttons.items():
        button.on_click(lambda _, s=s, k=k: choose(s, k))
    roll.on_click(roll_dice)
    paint()
    if not os.environ.get("ICM_BOOK_BUILD"):   # the build bakes no card
        render()
    return widgets.VBox([grid, widgets.HBox([roll, readout]), out])


icm_plotly.show(figure, controls)